# Hide a sphere, then recover it with symmetric chiselling
CC-BY Brooksbank, Kassabov, Wilson

This is an experiment you can change, not a prerecorded result. We will make a sphere-octant tensor, hide it with random orthogonal matrices, and recover its sparsity pattern. Then we will add a measured amount of noise and repeat **with the same sphere and hiding matrices**.

All construction, noise, plotting, and evaluation code is in this notebook. The numerical solver is part of Dleto: `stratify(...; method=:SymmetricGram)`.

Run cells in order from the repository root or `labs`. After changing the experiment controls, rerun the cells below them. A fresh kernel is needed after updating the package source.

In [ ]:
using Pkg
project_dir = isfile("Project.toml") ? pwd() : dirname(pwd())
Pkg.activate(project_dir)

using Dleto, ITensors, LinearAlgebra, Random, Plots
using Dleto.TensorSpace: ts
BLAS.set_num_threads(4)
gr()

## 1. Choose the experiment

Start with `d=50`, zero noise for the first solve, and `noise_level=1e-3` for the second. Here `1e-3` means **0.1% relative Frobenius noise**, not a per-entry standard deviation.

We sample uniformly in **squared radius**: $x_i=r\sqrt{i/(d-1)}$, for zero-based $i$. Then $x_i^2+x_j^2+x_k^2=r^2$ exactly when $i+j+k=d-1$. At $d=50$ this gives 1,275 points, with every slice represented. This is a sphere in physical coordinates, although its support is a plane in array indices.

A finite `shell_width` is measured in squared-distance units. It controls the *sample*, not numerical solver accuracy. Keep it at roundoff scale for the exact experiment.

In [ ]:
d = 50
radius = 1.0
shell_width = 1e-12 * radius^2
noise_level = 1e-3
amplitudes = :bounded              # try :gaussian to expose conditioning effects
sphere_seed = 50
hiding_seed = 10050
noise_seed = 51
solver_seed = 50
solver_tol = 1e-6                  # numerical solver accuracy, not shell width
requested_modes = 3               # two scalar modes + one geometric mode
plot_fraction = 0.20               # display cutoff only; metrics use every entry

x = radius .* sqrt.(collect(0:d-1) ./ (d-1))
@assert d >= 3 && radius > 0 && shell_width >= 0 && noise_level >= 0
@assert amplitudes in (:bounded, :gaussian)

## 2. Populate the sphere yourself

The loop below is the entire surface generator. A signed amplitude with magnitude between 0.5 and 1.5 keeps a populated boundary slice from becoming nearly zero. Try Gaussian amplitudes after the first successful run.

The center is at the origin and all coordinates are nonnegative, so this is one octant. A centered full sphere has repeated squared coordinates; rotations inside repeated eigenspaces introduce additional ambiguity.

In [ ]:
rng_sphere = MersenneTwister(sphere_seed)
A = zeros(Float64, d, d, d)
for i in 1:d, j in 1:d, k in 1:d
    squared_distance = x[i]^2 + x[j]^2 + x[k]^2
    if abs(squared_distance - radius^2) <= shell_width
        A[i,j,k] = amplitudes == :bounded ?
            (rand(rng_sphere, Bool) ? 1.0 : -1.0) * (0.5 + rand(rng_sphere)) :
            randn(rng_sphere)
    end
end

sphere_axes = [Index(d, "sphere_$a") for a in 1:3]
S = ts(ITensor(A, sphere_axes...))
occupied_slices = [count(i -> any(!iszero, selectdim(A, a, i)), 1:d) for a in 1:3]
@show count(!iszero, A) occupied_slices norm(A)
@assert all(==(d), occupied_slices) "Some slices are empty; this is no longer the full-slice experiment."

The plotting helper is deliberately visible. It only selects entries above a fraction of the largest amplitude and maps their indices to the supplied coordinates. It never changes the tensor passed to the solver.

In [ ]:
function show_points(B, coordinates; title, color=:blue, fraction=plot_fraction)
    cutoff = fraction * maximum(abs, B)
    chosen = findall(v -> abs(v) >= cutoff && !iszero(v), B)
    scatter([coordinates[1][I[1]] for I in chosen],
            [coordinates[2][I[2]] for I in chosen],
            [coordinates[3][I[3]] for I in chosen];
            title, color, markersize=2, markerstrokewidth=0, alpha=0.65,
            xlabel="x", ylabel="y", zlabel="z", legend=false,
            camera=(35,25), aspect_ratio=:equal)
end
show_points(A, (x,x,x); title="Original sphere octant")

## 3. Hide it with orthogonal changes of basis

`randomize_tensor` supplies one orthogonal matrix per axis. Orthogonal maps preserve the Frobenius norm and conjugate symmetric operators to symmetric operators. We test those properties explicitly.

The exact squared-coordinate sample has disjoint nonzero row supports in each mode unfolding, hence full mode rank. It needs no `nondeg` preprocessing. Avoid inserting a general invertible or whitening transform here: it changes the metric in which symmetry is defined.

In [ ]:
Random.seed!(hiding_seed)
S_hidden, Xs = randomize_tensor(S; type=:orthogonal)
@assert S * Xs ≈ S_hidden
@assert Dleto.is_orthogonal(Xs)

orthogonality_errors = [
    norm(Matrix(Array(X, inds(X)...))' * Matrix(Array(X, inds(X)...)) - I)
    for X in Xs
]
H = Array(S_hidden, inds(S_hidden)...)
@show orthogonality_errors norm(H) / norm(A)
plot(show_points(A, (x,x,x); title="Original", color=:blue),
     show_points(H, (1:d,1:d,1:d); title="Hidden: array coordinates", color=:orange);
     layout=(1,2), size=(1000,440))

## 4. Recover through the core command

The universal chisel is $[1\;1\;1]$. We explicitly restrict the operators to symmetric matrices, whose eigenvectors are orthogonal. Before hiding, the diagonal matrices with entries $x_i^2-r^2/3$ satisfy the derivation equation: their three contributions sum to zero on every occupied sphere entry. Orthogonal hiding conjugates these into dense symmetric matrices; stratification seeks their eigenbases.

`SymmetricGram` solves directly in that restricted space. With positive `nd`, it asks for a fixed number of smallest modes. For this one-surface model we request three: two scalar operators, which exist for every tensor, plus one geometric operator. **This is a model-order assumption, not a claim that noisy data has a three-dimensional exact kernel.**

`solver_tol` concerns numerical accuracy; it does not specify the noise level or choose how many geometric modes to retain. `:SymmetricGram` is opt-in; the default `:Auto` route remains unchanged. The first solve also pays Julia compilation cost.

In [ ]:
Ωsym = SymmetricOps(S_hidden)
P = UniversalChisel(3)
Random.seed!(solver_seed)           # reproducible combination of the returned modes
exact_run = @timed stratify(Ωsym, P, S_hidden;
    method=:SymmetricGram, nd=requested_modes, tol=solver_tol, seed=solver_seed)
S_recovered, Zs = exact_run.value
@assert S_hidden * Zs ≈ S_recovered
@show exact_run.time

## 5. Check recovery, not just the transformation identity

The identity above also holds for a bad basis. Here we use the known hiding matrices **only for evaluation**. Composing hiding and recovery should give a signed permutation on each axis. We extract that permutation, compare the full recovered tensor with the correspondingly reordered original, and measure energy on its support.

This alignment is an experimental oracle. The solver never receives `A`, `x`, or `Xs`. It also explains the physical coordinates in the recovered plot: arbitrary eigenvalue order is not a physical coordinate system.

The following helper spells out the complete measurement; there is no support-fitting routine in another folder.

In [ ]:
function measure_recovery(A, sphere_axes, Xs, recovered, Zs)
    permutations = Vector{Vector{Int}}()
    signs = Vector{Vector{Float64}}()
    output_axes = Index[]
    for original_axis in sphere_axes
        X = only(filter(M -> hasind(M, original_axis), Xs))
        hidden_axis = only(filter(!=(original_axis), collect(inds(X))))
        Z = only(filter(M -> hasind(M, hidden_axis), Zs))
        new_axis = only(filter(!=(hidden_axis), collect(inds(Z))))
        combined = Array(X * Z, original_axis, new_axis)
        permutation = [argmax(abs.(combined[:,j])) for j in axes(combined,2)]
        push!(permutations, permutation)
        push!(signs, [sign(combined[permutation[j],j]) for j in axes(combined,2)])
        push!(output_axes, hidden_axis) # stratify retags its output to the input frame
    end
    R = Array(recovered, output_axes...)
    reference = A[permutations...] .* reshape(signs[1],:,1,1) .*
                reshape(signs[2],1,:,1) .* reshape(signs[3],1,1,:)
    relative_error = norm(R - reference) / norm(A)
    support_energy = sum(abs2, R[reference .!= 0]) / sum(abs2, R)
    permutation_ok = all(allunique, permutations)
    (; R, permutations, relative_error, support_energy, permutation_ok)
end

exact_score = measure_recovery(A, sphere_axes, Xs, S_recovered, Zs)
@show exact_score.relative_error exact_score.support_energy exact_score.permutation_ok
@assert exact_score.permutation_ok && exact_score.relative_error < 1e-6
recovered_coordinates = Tuple(x[p] for p in exact_score.permutations)
plot(show_points(A, (x,x,x); title="Original", color=:blue),
     show_points(H, (1:d,1:d,1:d); title="Hidden: array coordinates", color=:orange),
     show_points(exact_score.R, recovered_coordinates;
                 title="Recovered: oracle-aligned coordinates", color=:green);
     layout=(1,3), size=(1500,460))

## 6. Add a controlled amount of noise

We draw one noise array and normalize it so $\|E\|_F=1$. Thus
$A_\eta=A+\eta\|A\|_F E$ has relative perturbation exactly $\eta$.

We reuse `Xs` and the same noise direction at every noise level. Changing `noise_level` now changes only the noise amplitude. Predict what should happen to the recovery error before running the next cells. Stratification changes the basis; it does not remove the added noise. We compare against the clean sphere, so the full-tensor error includes that noise.

In [ ]:
rng_noise = MersenneTwister(noise_seed)
E = randn(rng_noise, size(A)...)
E ./= norm(E)
A_noisy = A + noise_level * norm(A) * E
S_noisy = ts(ITensor(A_noisy, sphere_axes...))
S_hidden_noisy = S_noisy * Xs
@show norm(A_noisy - A) / norm(A)
@assert isapprox(norm(Array(S_hidden_noisy, inds(S_hidden)...) - H) / norm(H), noise_level; atol=1e-12)

Random.seed!(solver_seed)
noisy_run = @timed stratify(SymmetricOps(S_hidden_noisy), P, S_hidden_noisy;
    method=:SymmetricGram, nd=requested_modes, tol=solver_tol, seed=solver_seed)
S_recovered_noisy, Zs_noisy = noisy_run.value
@assert S_hidden_noisy * Zs_noisy ≈ S_recovered_noisy
noisy_score = measure_recovery(A, sphere_axes, Xs, S_recovered_noisy, Zs_noisy)
@show noisy_run.time noisy_score.relative_error noisy_score.support_energy noisy_score.permutation_ok

In [ ]:
noisy_coordinates = Tuple(x[p] for p in noisy_score.permutations)
plot(show_points(A, (x,x,x); title="Original signal", color=:blue),
     show_points(Array(S_hidden_noisy, inds(S_hidden_noisy)...), (1:d,1:d,1:d);
                 title="Hidden + noise: array coordinates", color=:orange),
     show_points(noisy_score.R, noisy_coordinates;
                 title="Recovered noisy tensor: oracle-aligned", color=:green);
     layout=(1,3), size=(1500,460))

## 7. Explore the balance

Set `run_sweep=true` to compare several noise levels. Each row is a new solve, so this is optional. The first solve above paid compilation costs; these times are more comparable, though they are still single runs.

Try changing one control at a time:

- Increase noise from $10^{-6}$ to $10^{-2}$. Does recovery error track the injected noise?
- Compare bounded and Gaussian support amplitudes, rerunning from section 2.
- Set `requested_modes=2` and rerun just the noisy solve in section 6 (keep the exact baseline at 3). Two scalar directions contain no sphere information.
- Change `solver_tol`. A tighter numerical solve cannot remove noise from the input or repair an incorrect mode count.
- Change `plot_fraction`. The picture changes, but the full-tensor scores must not.

A low residual for a scalar operator is not evidence of a recovered surface. A false `permutation_ok` is a failure of this signed-permutation recovery criterion.

In [ ]:
run_sweep = false
noise_levels = [0.0, 1e-6, 1e-3, 1e-2]
sweep_results = NamedTuple[]
if run_sweep
    for η in noise_levels
        trial = ts(ITensor(A + η * norm(A) * E, sphere_axes...)) * Xs
        Random.seed!(solver_seed)
        elapsed = @elapsed result = stratify(SymmetricOps(trial), P, trial;
            method=:SymmetricGram, nd=requested_modes, tol=solver_tol, seed=solver_seed)
        score = measure_recovery(A, sphere_axes, Xs, result.Σ, result.Xs)
        push!(sweep_results, (; noise=η, seconds=elapsed,
            error=score.relative_error, support=score.support_energy,
            permutation_ok=score.permutation_ok))
    end
end
sweep_results

## 8. Why a thin integer-grid sphere can be misleading

The previous SphereLab used integer coordinates with radius 48 inside a 50-cube and retained points within 1.5 **squared-distance units** of the sphere. Count them below before asking a solver to recover anything.

That sampling has only 48 entries and uses 19 slices per axis. It admits many diagonal derivations, so symmetry alone does not single out the intended sphere ordering. This is a sampling/identifiability issue, distinct from numerical accuracy. The squared-radius grid above deliberately supplies a richer, identifiable example.

A separate implementation issue found during the experiments was QuickDer's narrowed Gram stage losing symmetric directions. The new core method solves directly in the symmetric space. QuickSylver instead targets a two-axis chisel and does not support this experiment.

In [ ]:
integer_radius = d - 2
integer_support = [(i,j,k) for i in 0:d-1 for j in 0:d-1 for k in 0:d-1
    if abs(i^2 + j^2 + k^2 - integer_radius^2) < 1.5]
integer_active_slices = [length(unique(p[a] for p in integer_support)) for a in 1:3]
@show length(integer_support) integer_active_slices
# Try a wider shell and count again. More points also change the near-null problem.